# CaImAn Output Analysis

Explore and visualize outputs from the legacy caiman-matlab pipeline.

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import mbo_utilities as mbo
from ipywidgets import interact, IntSlider, Dropdown, Checkbox, IntRangeSlider

# configuration
output_dir = Path(r"//rbo-s1/S1_DATA/lbm/jdemas/bi_hemisphere/output")
raw_data_dir = Path(r"//rbo-s1/S1_DATA/lbm/jdemas/bi_hemisphere")
save_dir = None  # set to Path("./figures") to save outputs
plane_num = 1
zoom_size = 150  # default zoom region size in pixels

## Load data

In [ ]:
plane_file = output_dir / f"caiman_output_plane_{plane_num}.mat"

# show file structure
with h5py.File(plane_file, "r") as f:
    print(f"file: {plane_file.name}\n")
    print(f"{'key':<15} {'shape':<20} {'dtype'}")
    print("-" * 50)
    for key in sorted(f.keys()):
        item = f[key]
        if hasattr(item, "shape"):
            print(f"{key:<15} {str(item.shape):<20} {item.dtype}")

In [ ]:
def load_caiman_plane(filepath):
    """load all data from a caiman plane output file."""
    data = {}
    with h5py.File(filepath, "r") as f:
        for key in f.keys():
            try:
                arr = f[key][:]
                # transpose 2d image arrays (matlab is column-major)
                if arr.ndim == 2 and arr.shape[0] > 1 and arr.shape[1] > 1:
                    if 0.1 < arr.shape[0] / arr.shape[1] < 10:
                        arr = arr.T
                data[key] = arr
            except Exception as e:
                print(f"could not load {key}: {e}")
    return data


data = load_caiman_plane(plane_file)

# get fov shape from Cn or Ym
fov_shape = data.get("Cn", data.get("Ym")).shape
n_neurons = data["Ac_keep"].shape[0]
n_frames = data["T_keep"].shape[0]

print(f"\nloaded: {n_neurons:,} neurons, {n_frames} frames, fov {fov_shape}")

## Summary images (Cn and Ym)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, key in zip(axes, ["Cn", "Ym"]):
    if key in data:
        img = data[key]
        im = ax.imshow(img, cmap="gray",
                      vmin=np.percentile(img, 1),
                      vmax=np.percentile(img, 99))
        ax.set_title(f"{key}: {img.shape}")
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.show()

## Temporal traces (T_keep)

In [ ]:
T_keep = data["T_keep"]  # (n_frames, n_neurons)
print(f"T_keep shape: {T_keep.shape}")

# find most active neurons
mean_activity = T_keep.mean(axis=0)
top_neurons = np.argsort(mean_activity)[::-1][:50]

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# heatmap
ax = axes[0]
traces = T_keep[:, top_neurons].T
im = ax.imshow(traces, aspect="auto", cmap="viridis",
              vmin=np.percentile(traces, 5),
              vmax=np.percentile(traces, 95))
ax.set_xlabel("frame")
ax.set_ylabel("neuron (sorted by activity)")
ax.set_title("T_keep: top 50 most active neurons")
plt.colorbar(im, ax=ax)

# individual traces
ax = axes[1]
n_show = 15
offset = 0
for i in range(n_show):
    trace = T_keep[:, top_neurons[i]]
    trace_norm = (trace - trace.mean()) / (trace.std() + 1e-6)
    ax.plot(trace_norm + offset, lw=0.7, alpha=0.9)
    offset += 5
ax.set_xlabel("frame")
ax.set_ylabel("normalized (offset)")
ax.set_title(f"top {n_show} neurons")

plt.tight_layout()
plt.show()

## Reconstruct masks

In [ ]:
def reconstruct_masks(data, fov_shape):
    """reconstruct full-fov masks from cropped footprints and centroids."""
    ac = data["Ac_keep"]
    acx = data["acx"].flatten()
    acy = data["acy"].flatten()

    n_neurons = ac.shape[0]
    fp_size = ac.shape[1]
    half = fp_size // 2

    masks = []
    for i in range(n_neurons):
        fp = ac[i]
        cx, cy = int(acx[i]), int(acy[i])
        local_y, local_x = np.where(fp > 0)
        gx = local_x + cx - half
        gy = local_y + cy - half
        valid = (gx >= 0) & (gx < fov_shape[1]) & (gy >= 0) & (gy < fov_shape[0])

        masks.append({
            "x": gx[valid],
            "y": gy[valid],
            "weights": fp[local_y[valid], local_x[valid]],
            "cx": cx,
            "cy": cy,
        })
    return masks


masks = reconstruct_masks(data, fov_shape)
print(f"reconstructed {len(masks):,} masks")

## Interactive zoomed viewer with masks

Click and drag the center sliders to pan around the FOV. Masks are shown as semi-transparent colored overlays.

In [ ]:
def create_mask_overlay(masks, shape, region):
    """create colored mask overlay for a region."""
    y0, y1, x0, x1 = region
    h, w = y1 - y0, x1 - x0
    overlay = np.zeros((h, w, 4), dtype=np.float32)  # RGBA

    np.random.seed(42)
    colors = plt.cm.hsv(np.linspace(0, 1, 256))[:, :3]

    for i, m in enumerate(masks):
        if len(m["x"]) == 0:
            continue
        # check if mask overlaps region
        in_region = (m["x"] >= x0) & (m["x"] < x1) & (m["y"] >= y0) & (m["y"] < y1)
        if not np.any(in_region):
            continue

        lx = m["x"][in_region] - x0
        ly = m["y"][in_region] - y0
        weights = m["weights"][in_region]
        color = colors[i % 256]

        for j in range(len(lx)):
            alpha = min(0.6, weights[j] / (weights.max() + 1e-6))
            overlay[ly[j], lx[j], :3] = color
            overlay[ly[j], lx[j], 3] = max(overlay[ly[j], lx[j], 3], alpha)

    return overlay


def interactive_zoom(cy=None, cx=None, size=150, bg_image="Cn", show_masks=True, mask_alpha=0.5):
    """interactive zoomed viewer."""
    h, w = fov_shape
    if cy is None:
        cy = h // 2
    if cx is None:
        cx = w // 2

    half = size // 2
    y0 = max(0, cy - half)
    y1 = min(h, cy + half)
    x0 = max(0, cx - half)
    x1 = min(w, cx + half)
    region = (y0, y1, x0, x1)

    bg = data[bg_image]
    crop = bg[y0:y1, x0:x1]

    fig, ax = plt.subplots(figsize=(10, 10))

    # normalize background
    bg_norm = (crop - np.percentile(crop, 1)) / (np.percentile(crop, 99) - np.percentile(crop, 1) + 1e-6)
    bg_norm = np.clip(bg_norm, 0, 1)

    if show_masks:
        # create RGB from grayscale
        rgb = np.stack([bg_norm] * 3, axis=-1)

        # get mask overlay
        mask_overlay = create_mask_overlay(masks, fov_shape, region)

        # blend
        alpha = mask_overlay[:, :, 3:4] * mask_alpha
        rgb = rgb * (1 - alpha) + mask_overlay[:, :, :3] * alpha
        rgb = np.clip(rgb, 0, 1)

        ax.imshow(rgb)
    else:
        ax.imshow(crop, cmap="gray", vmin=np.percentile(crop, 1), vmax=np.percentile(crop, 99))

    # count masks in region
    n_in_region = sum(1 for m in masks if x0 <= m["cx"] < x1 and y0 <= m["cy"] < y1)

    ax.set_title(f"{bg_image} [{y0}:{y1}, {x0}:{x1}] - {n_in_region} neurons")
    ax.axis("off")
    plt.tight_layout()
    plt.show()


h, w = fov_shape
interact(
    interactive_zoom,
    cy=IntSlider(min=0, max=h-1, value=h//2, description="center y"),
    cx=IntSlider(min=0, max=w-1, value=w//2, description="center x"),
    size=IntSlider(min=50, max=500, step=25, value=zoom_size, description="size"),
    bg_image=Dropdown(options=["Cn", "Ym"], value="Cn", description="background"),
    show_masks=Checkbox(value=True, description="show masks"),
    mask_alpha=IntSlider(min=0, max=100, value=50, description="mask alpha") ,
)

## Load raw movie

In [ ]:
raw_movie = None

if raw_data_dir.exists():
    try:
        print(f"loading: {raw_data_dir}")
        raw_arr = mbo.imread(raw_data_dir)
        print(f"array: {type(raw_arr).__name__}, shape: {raw_arr.shape}")

        if raw_arr.ndim == 4:
            raw_movie = raw_arr[:, plane_num - 1, :, :]
            print(f"extracted plane {plane_num}: {raw_movie.shape}")
        else:
            raw_movie = raw_arr
    except Exception as e:
        print(f"error: {e}")
else:
    print(f"not found: {raw_data_dir}")

## Interactive movie viewer

Scroll through raw, reconstructed (A*C + b*f), and residual frames. Always zoomed to a region.

In [ ]:
def reconstruct_frame(data, frame_idx, fov_shape):
    """reconstruct frame from caiman: Y = A*C + b*f"""
    C = data["C_keep"]  # (n_frames, n_neurons)
    c_t = C[frame_idx, :]

    ac = data["Ac_keep"]
    acx = data["acx"].flatten()
    acy = data["acy"].flatten()

    n_neurons = ac.shape[0]
    fp_size = ac.shape[1]
    half = fp_size // 2

    neural_img = np.zeros(fov_shape, dtype=np.float32)

    for i in range(n_neurons):
        if c_t[i] == 0:
            continue
        fp = ac[i] * c_t[i]
        cx, cy = int(acx[i]), int(acy[i])

        y0 = cy - half
        y1 = cy - half + fp_size
        x0 = cx - half
        x1 = cx - half + fp_size

        fp_y0 = max(0, -y0)
        fp_y1 = fp_size - max(0, y1 - fov_shape[0])
        fp_x0 = max(0, -x0)
        fp_x1 = fp_size - max(0, x1 - fov_shape[1])

        y0 = max(0, y0)
        y1 = min(fov_shape[0], y1)
        x0 = max(0, x0)
        x1 = min(fov_shape[1], x1)

        if y1 > y0 and x1 > x0:
            neural_img[y0:y1, x0:x1] += fp[fp_y0:fp_y1, fp_x0:fp_x1]

    # background: b * f_t
    b = data["b"]
    f = data["f"]
    f_t = f[frame_idx, :]
    bg_flat = f_t @ b

    expected_pixels = fov_shape[0] * fov_shape[1]
    if bg_flat.shape[0] == expected_pixels:
        bg_img = bg_flat.reshape(fov_shape)
    else:
        bg_img = data.get("Ym", np.zeros(fov_shape))

    return neural_img + bg_img, neural_img, bg_img

In [ ]:
max_frames = min(n_frames, raw_movie.shape[0] if raw_movie is not None else n_frames)

def movie_viewer(frame=0, cy=None, cx=None, size=200):
    """view raw, reconstructed, and residual for a zoomed region."""
    h, w = fov_shape
    if cy is None:
        cy = h // 2
    if cx is None:
        cx = w // 2

    half = size // 2
    y0 = max(0, cy - half)
    y1 = min(h, cy + half)
    x0 = max(0, cx - half)
    x1 = min(w, cx + half)

    # get reconstructed
    recon, neural, bg = reconstruct_frame(data, frame, fov_shape)

    has_raw = raw_movie is not None
    n_panels = 4 if has_raw else 3

    fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 5))

    ax_idx = 0

    # raw
    if has_raw:
        raw_frame = np.asarray(raw_movie[frame]).astype(np.float32)
        # handle size mismatch
        if raw_frame.shape != fov_shape:
            min_h = min(raw_frame.shape[0], fov_shape[0])
            min_w = min(raw_frame.shape[1], fov_shape[1])
            tmp = np.zeros(fov_shape, dtype=np.float32)
            tmp[:min_h, :min_w] = raw_frame[:min_h, :min_w]
            raw_frame = tmp

        crop = raw_frame[y0:y1, x0:x1]
        ax = axes[ax_idx]
        im = ax.imshow(crop, cmap="gray", vmin=np.percentile(crop, 1), vmax=np.percentile(crop, 99))
        ax.set_title(f"raw (frame {frame})")
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046)
        ax_idx += 1

    # reconstructed
    crop = recon[y0:y1, x0:x1]
    ax = axes[ax_idx]
    im = ax.imshow(crop, cmap="gray", vmin=np.percentile(crop, 1), vmax=np.percentile(crop, 99))
    ax.set_title("reconstructed (A*C + b*f)")
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax_idx += 1

    # neural
    crop = neural[y0:y1, x0:x1]
    ax = axes[ax_idx]
    vmax = np.percentile(crop[crop > 0], 99) if np.any(crop > 0) else 1
    im = ax.imshow(crop, cmap="hot", vmin=0, vmax=vmax)
    ax.set_title("neural (A*C)")
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax_idx += 1

    # residual
    if has_raw:
        residual = raw_frame - recon
        crop = residual[y0:y1, x0:x1]
        ax = axes[ax_idx]
        vlim = np.percentile(np.abs(crop), 99)
        im = ax.imshow(crop, cmap="RdBu_r", vmin=-vlim, vmax=vlim)
        ax.set_title("residual (raw - recon)")
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046)

    plt.tight_layout()
    plt.show()


h, w = fov_shape
interact(
    movie_viewer,
    frame=IntSlider(min=0, max=max_frames-1, value=0, description="frame"),
    cy=IntSlider(min=0, max=h-1, value=h//2, description="center y"),
    cx=IntSlider(min=0, max=w-1, value=w//2, description="center x"),
    size=IntSlider(min=100, max=500, step=25, value=200, description="size"),
)

## Animated movie (zoomed region)

Play a sped-up video of a zoomed region to see neurons firing. Uses matplotlib animation.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def create_movie(cy, cx, size=150, start_frame=0, end_frame=200, speed=3):
    """
    create animated movie of a zoomed region.

    speed: frame step (1=every frame, 3=every 3rd frame, etc.)
    """
    h, w = fov_shape
    half = size // 2
    y0 = max(0, cy - half)
    y1 = min(h, cy + half)
    x0 = max(0, cx - half)
    x1 = min(w, cx + half)

    # match raw movie size if needed
    if raw_movie is not None:
        raw_h, raw_w = raw_movie.shape[1], raw_movie.shape[2]
        y1_raw = min(y1, raw_h)
        x1_raw = min(x1, raw_w)
    else:
        y1_raw, x1_raw = y1, x1

    frames_to_show = list(range(start_frame, min(end_frame, max_frames), speed))

    # precompute percentiles from first frame for consistent scaling
    if raw_movie is not None:
        sample = np.asarray(raw_movie[start_frame])[y0:y1_raw, x0:x1_raw].astype(np.float32)
        vmin, vmax = np.percentile(sample, 1), np.percentile(sample, 99)

    fig, ax = plt.subplots(figsize=(8, 8))

    # initial frame
    if raw_movie is not None:
        frame_data = np.asarray(raw_movie[start_frame])[y0:y1_raw, x0:x1_raw].astype(np.float32)
    else:
        frame_data = data["Ym"][y0:y1, x0:x1]
        vmin, vmax = np.percentile(frame_data, 1), np.percentile(frame_data, 99)

    im = ax.imshow(frame_data, cmap="gray", vmin=vmin, vmax=vmax, animated=True)
    title = ax.set_title(f"frame {start_frame}")
    ax.axis("off")
    plt.tight_layout()

    def update(frame_idx):
        if raw_movie is not None:
            frame_data = np.asarray(raw_movie[frame_idx])[y0:y1_raw, x0:x1_raw].astype(np.float32)
        else:
            # fallback to static image
            frame_data = data["Ym"][y0:y1, x0:x1]
        im.set_array(frame_data)
        title.set_text(f"frame {frame_idx}")
        return [im, title]

    anim = FuncAnimation(fig, update, frames=frames_to_show, interval=50, blit=True)
    plt.close(fig)
    return anim

# create animation - adjust cy, cx to center on region of interest
h, w = fov_shape
anim = create_movie(
    cy=h//2,           # center y
    cx=w//2,           # center x
    size=200,          # zoom size in pixels
    start_frame=0,     # start frame
    end_frame=300,     # end frame
    speed=2,           # skip frames (1=all, 2=every other, etc)
)

HTML(anim.to_jshtml())

## Save and play video

Export zoomed region as mp4 video using `mbo_utilities.to_video`.

In [ ]:
from mbo_utilities import to_video
from IPython.display import Video

# video settings
video_cy = fov_shape[0] // 2  # center y
video_cx = fov_shape[1] // 2  # center x
video_size = 200              # zoom size
video_frames = (0, 300)       # (start, end) frames
video_fps = 30                # playback fps
video_path = output_dir / f"plane{plane_num}_zoomed.mp4"

# extract zoomed region
h, w = fov_shape
half = video_size // 2
y0 = max(0, video_cy - half)
y1 = min(h, video_cy + half)
x0 = max(0, video_cx - half)
x1 = min(w, video_cx + half)

# handle raw movie size mismatch
if raw_movie is not None:
    raw_h, raw_w = raw_movie.shape[1], raw_movie.shape[2]
    y1 = min(y1, raw_h)
    x1 = min(x1, raw_w)

print(f"extracting frames {video_frames[0]}-{video_frames[1]}, region [{y0}:{y1}, {x0}:{x1}]...")

# load frames into array
frames = []
for i in range(video_frames[0], min(video_frames[1], max_frames)):
    if raw_movie is not None:
        frame = np.asarray(raw_movie[i])[y0:y1, x0:x1].astype(np.float32)
    else:
        # fallback to Ym repeated
        frame = data["Ym"][y0:y1, x0:x1].astype(np.float32)
    frames.append(frame)

frames = np.stack(frames)
print(f"video array shape: {frames.shape}")

# normalize to 0-1
lo, hi = np.percentile(frames, [1, 99])
frames_norm = np.clip((frames - lo) / (hi - lo + 1e-6), 0, 1)

# save video
to_video(frames_norm, video_path, fps=video_fps, cmap="gray")
print(f"saved: {video_path}")

In [ ]:
# play video in notebook
Video(video_path, embed=True, width=500)

In [ ]:
# side-by-side: raw + neural signal animation
def create_comparison_movie(cy, cx, size=150, start_frame=0, end_frame=200, speed=3):
    """
    create side-by-side animation: raw data vs reconstructed neural signal.
    """
    h, w = fov_shape
    half = size // 2
    y0 = max(0, cy - half)
    y1 = min(h, cy + half)
    x0 = max(0, cx - half)
    x1 = min(w, cx + half)

    if raw_movie is not None:
        raw_h, raw_w = raw_movie.shape[1], raw_movie.shape[2]
        y1_raw = min(y1, raw_h)
        x1_raw = min(x1, raw_w)
    else:
        y1_raw, x1_raw = y1, x1

    frames_to_show = list(range(start_frame, min(end_frame, max_frames), speed))

    # get scaling from first frame
    if raw_movie is not None:
        sample = np.asarray(raw_movie[start_frame])[y0:y1_raw, x0:x1_raw].astype(np.float32)
        raw_vmin, raw_vmax = np.percentile(sample, 1), np.percentile(sample, 99)

    # get neural scaling from a sample
    _, neural_sample, _ = reconstruct_frame(data, start_frame, fov_shape)
    neural_crop = neural_sample[y0:y1, x0:x1]
    neural_vmax = np.percentile(neural_crop[neural_crop > 0], 99) if np.any(neural_crop > 0) else 1

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))

    # initial frames
    if raw_movie is not None:
        raw_data = np.asarray(raw_movie[start_frame])[y0:y1_raw, x0:x1_raw].astype(np.float32)
    else:
        raw_data = data["Ym"][y0:y1, x0:x1]
        raw_vmin, raw_vmax = np.percentile(raw_data, 1), np.percentile(raw_data, 99)

    im_raw = axes[0].imshow(raw_data, cmap="gray", vmin=raw_vmin, vmax=raw_vmax, animated=True)
    axes[0].set_title("raw")
    axes[0].axis("off")

    im_neural = axes[1].imshow(neural_crop, cmap="hot", vmin=0, vmax=neural_vmax, animated=True)
    axes[1].set_title("neural (A*C)")
    axes[1].axis("off")

    title = fig.suptitle(f"frame {start_frame}")
    plt.tight_layout()

    def update(frame_idx):
        # raw
        if raw_movie is not None:
            raw_data = np.asarray(raw_movie[frame_idx])[y0:y1_raw, x0:x1_raw].astype(np.float32)
        else:
            raw_data = data["Ym"][y0:y1, x0:x1]
        im_raw.set_array(raw_data)

        # neural
        _, neural, _ = reconstruct_frame(data, frame_idx, fov_shape)
        neural_crop = neural[y0:y1, x0:x1]
        im_neural.set_array(neural_crop)

        title.set_text(f"frame {frame_idx}")
        return [im_raw, im_neural, title]

    anim = FuncAnimation(fig, update, frames=frames_to_show, interval=50, blit=False)
    plt.close(fig)
    return anim

# comparison animation - raw vs neural signal
h, w = fov_shape
anim2 = create_comparison_movie(
    cy=h//2,
    cx=w//2,
    size=200,
    start_frame=0,
    end_frame=200,
    speed=2,
)

HTML(anim2.to_jshtml())

## Interactive neuron viewer

Select a neuron to see its mask on the background and temporal trace.

In [ ]:
def view_neuron(neuron_idx=0, context_size=80, bg_image="Cn"):
    """view a single neuron's mask and trace."""
    ac = data["Ac_keep"]
    acx = data["acx"].flatten()
    acy = data["acy"].flatten()
    T = data["T_keep"]

    fp = ac[neuron_idx]
    cx, cy = int(acx[neuron_idx]), int(acy[neuron_idx])
    trace = T[:, neuron_idx]

    bg = data[bg_image]

    half = context_size
    y0 = max(0, cy - half)
    y1 = min(fov_shape[0], cy + half)
    x0 = max(0, cx - half)
    x1 = min(fov_shape[1], cx + half)

    crop = bg[y0:y1, x0:x1]

    fig = plt.figure(figsize=(14, 5))
    gs = fig.add_gridspec(1, 2, width_ratios=[1, 2])

    # spatial: background with mask overlay
    ax = fig.add_subplot(gs[0])

    # normalize background
    bg_norm = (crop - np.percentile(crop, 1)) / (np.percentile(crop, 99) - np.percentile(crop, 1) + 1e-6)
    bg_norm = np.clip(bg_norm, 0, 1)
    rgb = np.stack([bg_norm] * 3, axis=-1)

    # create mask overlay
    fp_half = fp.shape[0] // 2
    local_cy = cy - y0
    local_cx = cx - x0

    mask = np.zeros((y1 - y0, x1 - x0), dtype=np.float32)
    for ly in range(fp.shape[0]):
        for lx in range(fp.shape[1]):
            gy = local_cy - fp_half + ly
            gx = local_cx - fp_half + lx
            if 0 <= gy < mask.shape[0] and 0 <= gx < mask.shape[1]:
                mask[gy, gx] = fp[ly, lx]

    # blend mask (red) with background
    mask_norm = mask / (mask.max() + 1e-6)
    alpha = mask_norm * 0.6
    rgb[:, :, 0] = rgb[:, :, 0] * (1 - alpha) + alpha  # red channel
    rgb[:, :, 1] = rgb[:, :, 1] * (1 - alpha * 0.5)  # dim green
    rgb[:, :, 2] = rgb[:, :, 2] * (1 - alpha * 0.5)  # dim blue

    ax.imshow(np.clip(rgb, 0, 1))
    ax.plot(local_cx, local_cy, "w+", markersize=12, mew=2)
    ax.set_title(f"neuron {neuron_idx}\ncentroid: ({cx}, {cy})")
    ax.axis("off")

    # temporal trace
    ax = fig.add_subplot(gs[1])
    ax.plot(trace, "b-", lw=0.8)
    ax.set_xlabel("frame")
    ax.set_ylabel("activity")
    ax.set_title(f"T_keep trace - mean={trace.mean():.2f}, max={trace.max():.2f}")
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


interact(
    view_neuron,
    neuron_idx=IntSlider(min=0, max=n_neurons-1, value=top_neurons[0], description="neuron"),
    context_size=IntSlider(min=30, max=200, step=10, value=80, description="context"),
    bg_image=Dropdown(options=["Cn", "Ym"], value="Cn", description="background"),
)

## All planes summary

In [ ]:
plane_stats = []

for i in range(1, 31):
    pf = output_dir / f"caiman_output_plane_{i}.mat"
    if pf.exists():
        try:
            with h5py.File(pf, "r") as f:
                km = int(f["Km"][0, 0])
                plane_stats.append({"plane": i, "neurons": km})
        except Exception as e:
            print(f"plane {i}: {e}")

if plane_stats:
    total = sum(p["neurons"] for p in plane_stats)
    print(f"{len(plane_stats)} planes, {total:,} total neurons")

    fig, ax = plt.subplots(figsize=(12, 4))
    planes = [p["plane"] for p in plane_stats]
    counts = [p["neurons"] for p in plane_stats]
    ax.bar(planes, counts, color="steelblue", edgecolor="black")
    ax.set_xlabel("plane")
    ax.set_ylabel("neurons")
    ax.set_title(f"neurons per plane (total: {total:,})")
    ax.set_xticks(planes)
    plt.tight_layout()
    plt.show()